# Architecture Depth Study

This notebook performs experiments to find out the influence that convolutional layer brings to model.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from google.colab import drive



drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/CA Notebook/image"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR  = os.path.join(BASE_DIR, "test")


Mounted at /content/drive


global random seed

In [ ]:

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True # Ensure deterministic convolution algorithms
    torch.backends.cudnn.benchmark = False

seed_everything(42) # Lock random seed for reproducibility

load dataset

In [ ]:
from torch.utils.data import Dataset
class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        '''
        Custom dataset for fruit classification.
        Image label is inferred from filename prefix.
        '''
        self.image_files = [f for f in os.listdir(root_dir) if f.lower().endswith(".jpg")]
        self.class_map = {"apple": 0, "banana": 1, "orange": 2, "mixed": 3}
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.root_dir, filename)
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Label parsing logic
        label = -1
        for cls, idx_val in self.class_map.items():
            if filename.lower().startswith(cls):
                label = idx_val
                break
        return image, label

def get_loaders(root_dir, img_size, batch_size):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    # Load the full dataset first for split
    full_dataset = FruitDataset(root_dir, transform=transform)
    indices = list(range(len(full_dataset)))

    # Extract labels for Stratified Split
    labels = []
    for f in full_dataset.image_files:
        for cls, idx in full_dataset.class_map.items():
            if f.lower().startswith(cls):
                labels.append(idx)
                break

    # Split into 80% training and 20% validation set
    train_idx, val_idx = train_test_split(
        indices, test_size=0.2, stratify=labels, random_state=42
    )

    train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

CNN definition

In [ ]:
class DeepFruitCNN(nn.Module):
    def __init__(self, num_layers=2, num_classes=4):
        super(DeepFruitCNN, self).__init__()

        layers = []
        in_channels = 3
        # Filter sequence for selecting different network depths
        filters = [16, 32, 64, 128]

        for i in range(num_layers):
            layers.append(nn.Conv2d(in_channels, filters[i], kernel_size=3, padding=1))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool2d(2, 2))
            in_channels = filters[i]

        self.conv_section = nn.Sequential(*layers)

        # AdaptiveAvgPool2d ensures fixed input dimensions for the fully connected layer, independent of the number of convolutional layers
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        self.fc = nn.Sequential(
            nn.Linear(filters[num_layers-1] * 4 * 4, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_section(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

training process

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return total_loss / len(dataloader), correct / total

def evaluate(model, dataloader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

In [ ]:
import matplotlib.pyplot as plt

# Global dictionary for storing all experiment results
# Format: { num_layers: { 'train_loss': [], 'train_acc': [], 'val_acc': [] , 'test_acc': 0.0 } }
if 'all_results' not in locals():
    all_results = {}

print("experiment memory is ready.")

experiment memory is ready.


3 layer

In [ ]:
# ================= Manual Modification Area =================
N_LAYERS = 3           # <--- Modify the number of layers here for testing
EPOCHS = 30            # Train epochs
CURRENT_SIZE = 64      # Fixed image size
BATCH_SIZE = 16
# ===============================================

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_everything(42) # Ensure consistent weight initialization for each manual run

# Prepare data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])), batch_size=BATCH_SIZE, shuffle=False)

# Initialize the model
model = DeepFruitCNN(num_layers=N_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Initialize record for the current number of layers
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print(f"\n🚀 Start testing {N_LAYERS} layer convolutional neural network...")

for epoch in range(EPOCHS):
    # Training step
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    # Validation step
    v_acc = evaluate(model, val_loader)

    # Save training progress
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    # --- Print detailed results for each epoch ---
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Loss: {t_loss:.4f} | "
          f"Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f}")

# Final test on test set
final_test_acc = evaluate(model, test_loader)
history['test_acc'] = final_test_acc

# Save results to the global dictionary
all_results[N_LAYERS] = history

print(f"\n✅ {N_LAYERS} layer experiment finished！")
print(f"Final Test Accuracy: {final_test_acc:.4f}")


🚀 Start testing 3 layer convolutional neural network...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3451 | Train Acc: 0.2604 | Val Acc: 0.5833
Epoch [02/30] Loss: 1.2483 | Train Acc: 0.5052 | Val Acc: 0.3125
Epoch [03/30] Loss: 1.0023 | Train Acc: 0.6458 | Val Acc: 0.7917
Epoch [04/30] Loss: 0.7449 | Train Acc: 0.7240 | Val Acc: 0.7500
Epoch [05/30] Loss: 0.5213 | Train Acc: 0.8281 | Val Acc: 0.8333
Epoch [06/30] Loss: 0.3705 | Train Acc: 0.8646 | Val Acc: 0.8333
Epoch [07/30] Loss: 0.4129 | Train Acc: 0.8333 | Val Acc: 0.8333
Epoch [08/30] Loss: 0.2662 | Train Acc: 0.8958 | Val Acc: 0.8750
Epoch [09/30] Loss: 0.2124 | Train Acc: 0.9375 | Val Acc: 0.9167
Epoch [10/30] Loss: 0.1724 | Train Acc: 0.9375 | Val Acc: 0.8750
Epoch [11/30] Loss: 0.1435 | Train Acc: 0.9427 | Val Acc: 0.9167
Epoch [12/30] Loss: 0.1169 | Train Acc: 0.9531 | Val Acc: 0.8958
Epoch [13/30] Loss: 0.1094 | Train Acc: 0.9583 | Val Acc: 0.9167
Epoch [14/30] Loss: 0.0861 | Train Acc: 0.9740 | Val Acc: 0.9167
Epoch [15/30] Loss: 0.0750 | Train Acc: 0.9688 | Val Acc: 0.9167
Epoch [16/30] Loss: 0.100

2 layer

In [ ]:
# ================= Manual Modification Area =================
N_LAYERS = 2           # <--- Modify the number of layers here for testing
EPOCHS = 30            # Train epochs
CURRENT_SIZE = 64      # Fixed image size
BATCH_SIZE = 16
# ===============================================

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_everything(42) # Ensure consistent weight initialization for each manual run

# Prepare data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])), batch_size=BATCH_SIZE, shuffle=False)

# Initialize the model
model = DeepFruitCNN(num_layers=N_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Initialize record for the current number of layers
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print(f"\n🚀 Start testing {N_LAYERS} layer convolutional neural network...")

for epoch in range(EPOCHS):
    # Training step
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    # Validation step
    v_acc = evaluate(model, val_loader)

    # Save training progress
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    # --- Print detailed results for each epoch ---
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Loss: {t_loss:.4f} | "
          f"Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f}")

# Final test on test set
final_test_acc = evaluate(model, test_loader)
history['test_acc'] = final_test_acc

# Save results to the global dictionary
all_results[N_LAYERS] = history

print(f"\n✅ {N_LAYERS} layer experiment finished！")
print(f"Final Test Accuracy: {final_test_acc:.4f}")


🚀 Start testing 2 layer convolutional neural network...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3466 | Train Acc: 0.2708 | Val Acc: 0.5000
Epoch [02/30] Loss: 1.2941 | Train Acc: 0.4062 | Val Acc: 0.6667
Epoch [03/30] Loss: 1.2348 | Train Acc: 0.5052 | Val Acc: 0.5625
Epoch [04/30] Loss: 1.0791 | Train Acc: 0.6771 | Val Acc: 0.6458
Epoch [05/30] Loss: 0.8861 | Train Acc: 0.6562 | Val Acc: 0.7917
Epoch [06/30] Loss: 0.6937 | Train Acc: 0.7812 | Val Acc: 0.7500
Epoch [07/30] Loss: 0.5805 | Train Acc: 0.8125 | Val Acc: 0.8542
Epoch [08/30] Loss: 0.4701 | Train Acc: 0.8177 | Val Acc: 0.8542
Epoch [09/30] Loss: 0.4565 | Train Acc: 0.8229 | Val Acc: 0.8333
Epoch [10/30] Loss: 0.3789 | Train Acc: 0.8698 | Val Acc: 0.8750
Epoch [11/30] Loss: 0.3853 | Train Acc: 0.8542 | Val Acc: 0.8542
Epoch [12/30] Loss: 0.3469 | Train Acc: 0.8958 | Val Acc: 0.8958
Epoch [13/30] Loss: 0.2813 | Train Acc: 0.8958 | Val Acc: 0.8333
Epoch [14/30] Loss: 0.2463 | Train Acc: 0.9167 | Val Acc: 0.8750
Epoch [15/30] Loss: 0.2121 | Train Acc: 0.9271 | Val Acc: 0.8542
Epoch [16/30] Loss: 0.198

4 layer

In [ ]:
# ================= Manual Modification Area =================
N_LAYERS = 4           # <--- Modify the number of layers here for testing
EPOCHS = 30            # Train epochs
CURRENT_SIZE = 64      # Fixed image size
BATCH_SIZE = 16
# ===============================================

# Define the device (CPU or GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed_everything(42) # Ensure consistent weight initialization for each manual run

# Prepare data
train_loader, val_loader = get_loaders(TRAIN_DIR, CURRENT_SIZE, BATCH_SIZE)
test_loader = DataLoader(FruitDataset(TEST_DIR, transform=transforms.Compose([
    transforms.Resize((CURRENT_SIZE, CURRENT_SIZE)),
    transforms.ToTensor()
])), batch_size=BATCH_SIZE, shuffle=False)

# Initialize the model
model = DeepFruitCNN(num_layers=N_LAYERS).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Initialize record for the current number of layers
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}

print(f"\n🚀 Start testing {N_LAYERS} layer convolutional neural network...")

for epoch in range(EPOCHS):
    # Training step
    t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    # Validation step
    v_acc = evaluate(model, val_loader)

    # Save training progress
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    # --- Print detailed results for each epoch ---
    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Loss: {t_loss:.4f} | "
          f"Train Acc: {t_acc:.4f} | "
          f"Val Acc: {v_acc:.4f}")

# Final test on test set
final_test_acc = evaluate(model, test_loader)
history['test_acc'] = final_test_acc

# Save results to the global dictionary
all_results[N_LAYERS] = history

print(f"\n✅ {N_LAYERS} layer experiment finished！")
print(f"Final Test Accuracy: {final_test_acc:.4f}")


🚀 Start testing 4 layer convolutional neural network...


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [01/30] Loss: 1.3463 | Train Acc: 0.3073 | Val Acc: 0.5000
Epoch [02/30] Loss: 1.2904 | Train Acc: 0.5312 | Val Acc: 0.5208
Epoch [03/30] Loss: 1.0249 | Train Acc: 0.6094 | Val Acc: 0.5417
Epoch [04/30] Loss: 0.7684 | Train Acc: 0.7031 | Val Acc: 0.7917
Epoch [05/30] Loss: 0.5479 | Train Acc: 0.7865 | Val Acc: 0.7708
Epoch [06/30] Loss: 0.4076 | Train Acc: 0.8698 | Val Acc: 0.8958
Epoch [07/30] Loss: 0.3250 | Train Acc: 0.9115 | Val Acc: 0.8542
Epoch [08/30] Loss: 0.2946 | Train Acc: 0.8958 | Val Acc: 0.9375
Epoch [09/30] Loss: 0.1932 | Train Acc: 0.9323 | Val Acc: 0.8542
Epoch [10/30] Loss: 0.2680 | Train Acc: 0.9010 | Val Acc: 0.9375
Epoch [11/30] Loss: 0.1645 | Train Acc: 0.9323 | Val Acc: 0.9375
Epoch [12/30] Loss: 0.1403 | Train Acc: 0.9635 | Val Acc: 0.9583
Epoch [13/30] Loss: 0.1556 | Train Acc: 0.9479 | Val Acc: 0.9583
Epoch [14/30] Loss: 0.1151 | Train Acc: 0.9531 | Val Acc: 0.9375
Epoch [15/30] Loss: 0.1027 | Train Acc: 0.9635 | Val Acc: 0.9583
Epoch [16/30] Loss: 0.063

result comparison

In [ ]:
def plot_final_comparison(results):
    if not results:
        print("No experiment result found！")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    for layers, data in sorted(results.items()):
        epochs = range(1, len(data['val_acc']) + 1)

        # left：Train Loss
        ax1.plot(epochs, data['train_loss'], label=f'{layers} Layers (Test: {data["test_acc"]:.2f})')
        # right：Validate Accuracy
        ax2.plot(epochs, data['val_acc'], label=f'{layers} Layers')

    ax1.set_title('Training Loss Comparison')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title('Validation Accuracy Comparison')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# call the plot func
plot_final_comparison(all_results)

we choose 3 convolutional layer cnn as the base architecture.